In [ ]:
from pathlib import Path

import trimesh
import numpy as np
import open3d as o3d
# 加载三个模型
mesh1 = trimesh.load('../assets/basic_geo/3/3.obj')  # 替换为实际路径
mesh2 = trimesh.load('../assets/basic_geo/2/2.obj')  # 替换为实际路径
mesh3 = trimesh.load('../assets/basic_geo/3/3.obj')  # 替换为实际路径

# 对第一个模型进行平移和旋转
translation1 = [20, 0, 0]  # x, y, z 平移
rotation_matrix1 = trimesh.transformations.rotation_matrix(
    angle=np.radians(30),  # 旋转角度
    direction=[0, 1, 0],   # 绕 y 轴旋转
    point=[0, 0, 0]        # 旋转中心
)
mesh1.apply_transform(rotation_matrix1)
mesh1.apply_translation(translation1)

# 对第二个模型进行平移和旋转
translation2 = [-40, 0, 30]
rotation_matrix2 = trimesh.transformations.rotation_matrix(
    angle=np.radians(45),
    direction=[0, 1, 0],   # 绕 y 轴旋转
    point=[0, 0, 0]
)
mesh2.apply_transform(rotation_matrix2)
mesh2.apply_translation(translation2)

# 对第三个模型进行平移和旋转
translation3 = [70, 0, -60]
rotation_matrix3 = trimesh.transformations.rotation_matrix(
    angle=np.radians(60),
    direction=[0, 1, 0],   # 绕 y 轴旋转
    point=[0, 0, 0]
)
mesh3.apply_transform(rotation_matrix3)
mesh3.apply_translation(translation3)

# 合并三个模型
merged_mesh = trimesh.util.concatenate([mesh1, mesh2, mesh3])

# 保存拼接后的模型
merged_mesh.export('./merged_trapezoid_mesh.obj')  # 替换为目标路径


print("模型梯形拼接完成，已保存为 merged_trapezoid_mesh.obj")

# transform, _ = trimesh.registration.icp(mesh1.vertices, mesh2.vertices,mesh3.vertices)
# mesh1.apply_transform(transform)  # 应用变换但仍是独立对象

# 可视化（保持独立）
scene = trimesh.Scene([mesh1, mesh2])
scene.show()


mesh1 = o3d.io.read_triangle_mesh(Path('../assets/basic_geo/3/3.obj'))  # 替换为实际路径
mesh2 = o3d.io.read_triangle_mesh(Path('../assets/basic_geo/2/2.obj'))  # 替换为实际路径
mesh3 = o3d.io.read_triangle_mesh(Path('../assets/basic_geo/3/3.obj'))  # 替换为实际路径

# 对第一个模型进行平移和旋转


mesh1.transform(rotation_matrix1)
mesh1.translate(translation1)

mesh2.transform(rotation_matrix2)
mesh2.translate(translation2)

mesh3.transform(rotation_matrix3)
mesh3.translate(translation3)

pcd1 = mesh1.sample_points_uniformly(number_of_points=1000)
pcd2 = mesh2.sample_points_uniformly(number_of_points=1000)
pcd3 = mesh3.sample_points_uniformly(number_of_points=1000)

init_transform = np.eye(4)

# 调整参数示例
result = o3d.pipelines.registration.registration_icp(
    pcd1, pcd2,
    max_correspondence_distance=20.0,  # 增大最大距离阈值
    init=np.eye(4),                   # 初始化为单位矩阵（无初始变换）
    estimation_method=o3d.pipelines.registration.TransformationEstimationPointToPlane(),
    criteria=o3d.pipelines.registration.ICPConvergenceCriteria(
        max_iteration=1000,           # 增加迭代次数
        relative_fitness=1e-6,       # 更严格的收敛条件
        relative_rmse=1e-6
    )
)
mesh1.transform(result.transformation)

# 可视化
o3d.visualization.draw_geometries([mesh1, mesh2])




In [27]:
import open3d as o3d
import trimesh
import numpy as np
from pathlib import Path
from pycpd import DeformableRegistration

translation1 = [20, 0, 0]  # x, y, z 平移
rotation_matrix1 = trimesh.transformations.rotation_matrix(
    angle=np.radians(30),  # 旋转角度
    direction=[0, 1, 0],   # 绕 y 轴旋转
    point=[0, 0, 0]        # 旋转中心
)

# 对第二个模型进行平移和旋转
translation2 = [-40, 0, 30]
rotation_matrix2 = trimesh.transformations.rotation_matrix(
    angle=np.radians(45),
    direction=[0, 1, 0],   # 绕 y 轴旋转
    point=[0, 0, 0]
)


# 对第三个模型进行平移和旋转
translation3 = [70, 0, -60]
rotation_matrix3 = trimesh.transformations.rotation_matrix(
    angle=np.radians(60),
    direction=[0, 1, 0],   # 绕 y 轴旋转
    point=[0, 0, 0]
)

# 加载网格文件
mesh1 = o3d.io.read_triangle_mesh(Path('../assets/basic_geo/3/3.obj'))
mesh2 = o3d.io.read_triangle_mesh(Path('../assets/basic_geo/2/2.obj'))

mesh1.transform(rotation_matrix1)
mesh1.translate(translation1)

mesh2.transform(rotation_matrix2)
mesh2.translate(translation2)

# mesh3.transform(rotation_matrix3)
# mesh3.translate(translation3)

# 从网格采样点云
pcd1 = mesh1.sample_points_uniformly(number_of_points=1000)
pcd2 = mesh2.sample_points_uniformly(number_of_points=1000)

# 转换为 numpy 数组
source = np.asarray(pcd2.points)
target = np.asarray(pcd1.points)

# 定义非刚性配准的回调函数
def callback(iteration, error, X, Y):
    print(f"Iteration: {iteration}, Error: {error}")

# 执行非刚性配准
reg = DeformableRegistration(X=source, Y=target)
transformed_source, _ = reg.register(callback)

# 更新点云
pcd2.points = o3d.utility.Vector3dVector(transformed_source)

# 将变换应用到 mesh2
mesh2.vertices = o3d.utility.Vector3dVector(transformed_source)

# 合并点云
merged_pcd = pcd1 + pcd2

# 可视化合并后的点云
o3d.visualization.draw_geometries([merged_pcd, mesh1, mesh2])

# 合并点云
merged_pcd = pcd1 + pcd2

# 可视化合并后的点云
o3d.visualization.draw_geometries([merged_pcd])

# 合并网格
merged_mesh = mesh1 + mesh2

# 可视化合并后的网格
o3d.visualization.draw_geometries([merged_mesh])

[Open3D INFO] Skipping non-triangle primitive geometry of type: 2
Iteration: 1, Error: inf
Iteration: 2, Error: inf
Iteration: 3, Error: inf
Iteration: 4, Error: inf
Iteration: 5, Error: inf
Iteration: 6, Error: inf
Iteration: 7, Error: inf
Iteration: 8, Error: inf
Iteration: 9, Error: inf
Iteration: 10, Error: inf
Iteration: 11, Error: inf
Iteration: 12, Error: inf
Iteration: 13, Error: inf
Iteration: 14, Error: inf
Iteration: 15, Error: inf


In [5]:
import open3d as o3d
import numpy as np

import open3d as o3d
import trimesh
import numpy as np
from pathlib import Path
from pycpd import DeformableRegistration

translation1 = [20, 0, 0]  # x, y, z 平移
rotation_matrix1 = trimesh.transformations.rotation_matrix(
    angle=np.radians(30),  # 旋转角度
    direction=[0, 1, 0],   # 绕 y 轴旋转
    point=[0, 0, 0]        # 旋转中心
)

# 对第二个模型进行平移和旋转
translation2 = [-40, 0, 30]
rotation_matrix2 = trimesh.transformations.rotation_matrix(
    angle=np.radians(45),
    direction=[0, 1, 0],   # 绕 y 轴旋转
    point=[0, 0, 0]
)


# 对第三个模型进行平移和旋转
translation3 = [70, 0, -60]
rotation_matrix3 = trimesh.transformations.rotation_matrix(
    angle=np.radians(60),
    direction=[0, 1, 0],   # 绕 y 轴旋转
    point=[0, 0, 0]
)

# 加载网格文件
mesh1 = o3d.io.read_triangle_mesh(Path('../assets/basic_geo/3/3.obj'))
mesh2 = o3d.io.read_triangle_mesh(Path('../assets/basic_geo/2/2.obj'))

mesh1.transform(rotation_matrix1)
mesh1.translate(translation1)

mesh2.transform(rotation_matrix2)
mesh2.translate(translation2)

# mesh3.transform(rotation_matrix3)
# mesh3.translate(translation3)

# 从网格采样点云
pcd1 = mesh1.sample_points_uniformly(number_of_points=1000)
pcd2 = mesh2.sample_points_uniformly(number_of_points=1000)

# 1. 计算法线
pcd1.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30))
pcd2.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30))

# 2. 提取FPFH特征
fpfh1 = o3d.pipelines.registration.compute_fpfh_feature(
    pcd1, o3d.geometry.KDTreeSearchParamHybrid(radius=0.25, max_nn=100))
fpfh2 = o3d.pipelines.registration.compute_fpfh_feature(
    pcd2, o3d.geometry.KDTreeSearchParamHybrid(radius=0.25, max_nn=100))

# 3. 使用RANSAC进行初始变换估计
distance_threshold = 0.5
result_ransac = o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
    pcd1, pcd2, fpfh1, fpfh2, mutual_filter=True,  # 添加 mutual_filter 参数
    max_correspondence_distance=distance_threshold,
    estimation_method=o3d.pipelines.registration.TransformationEstimationPointToPoint(False),
    ransac_n=4,  # RANSAC采样点数
    checkers=[
        o3d.pipelines.registration.CorrespondenceCheckerBasedOnEdgeLength(0.9),
        o3d.pipelines.registration.CorrespondenceCheckerBasedOnDistance(distance_threshold)
    ],
    criteria=o3d.pipelines.registration.RANSACConvergenceCriteria(4000000, 500)
)

# 4. 使用ICP进行精细配准
result_icp = o3d.pipelines.registration.registration_icp(
    pcd1, pcd2, max_correspondence_distance=0.2,
    init=result_ransac.transformation,
    estimation_method=o3d.pipelines.registration.TransformationEstimationPointToPlane())

# 5. 应用变换到点云
pcd2.transform(result_icp.transformation)

# 6. 合并点云
merged_pcd = pcd1 + pcd2

# 可视化结果
o3d.visualization.draw_geometries([merged_pcd])

[Open3D INFO] Skipping non-triangle primitive geometry of type: 2
[Open3D WARNING] Too few correspondences (6) after mutual filter, fall back to original correspondences.


In [ ]:
import open3d as o3d
import numpy as np
import trimesh
from pathlib import Path
from scipy.spatial import KDTree

def load_and_position_meshes():
    """加载并定位初始网格"""
    mesh1 = o3d.io.read_triangle_mesh(Path('../assets/basic_geo/3/3.obj'))
    mesh2 = o3d.io.read_triangle_mesh(Path('../assets/basic_geo/2/2.obj'))

    # 初始变换矩阵
    trans1 = np.eye(4)
    trans1[:3, 3] = [20, 0, 0]
    rot1 = trimesh.transformations.rotation_matrix(np.radians(30), [0, 1, 0])

    trans2 = np.eye(4)
    trans2[:3, 3] = [-40, 0, 30]
    rot2 = trimesh.transformations.rotation_matrix(np.radians(45), [0, 1, 0])

    mesh1.transform(rot1 @ trans1)
    mesh2.transform(rot2 @ trans2)

    return mesh1, mesh2

def feature_based_registration(mesh1, mesh2):
    """基于特征的粗配准"""
    pcd1 = mesh1.sample_points_uniformly(1000)
    pcd2 = mesh2.sample_points_uniformly(1000)

    # 计算法线和特征
    pcd1.estimate_normals()
    pcd2.estimate_normals()
    fpfh1 = o3d.pipelines.registration.compute_fpfh_feature(
        pcd1, o3d.geometry.KDTreeSearchParamHybrid(radius=0.25, max_nn=100))
    fpfh2 = o3d.pipelines.registration.compute_fpfh_feature(
        pcd2, o3d.geometry.KDTreeSearchParamHybrid(radius=0.25, max_nn=100))

    # RANSAC配准
    result = o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
        pcd1, pcd2, fpfh1, fpfh2, mutual_filter=True,
        max_correspondence_distance=0.5,
        estimation_method=o3d.pipelines.registration.TransformationEstimationPointToPoint(False),
        ransac_n=4,
        criteria=o3d.pipelines.registration.RANSACConvergenceCriteria(4000000, 500))

    return result.transformation

def constrained_deformation(mesh1, mesh2, transform, scale=0.8):
    """带接触约束的变形"""
    # 应用初始配准
    mesh2.transform(transform)

    # 检测接触区域
    pcd1 = mesh1.sample_points_uniformly(1000)
    pcd2 = mesh2.sample_points_uniformly(1000)
    kdtree = KDTree(np.asarray(pcd2.points))
    dists, _ = kdtree.query(np.asarray(pcd1.points))
    contact_idx = np.where(dists < 0.05)[0]

    if len(contact_idx) < 10:
        print("警告：接触点不足，使用简单缩放")
        mesh2.scale(scale, center=mesh2.get_center())
        return mesh1, mesh2

    # 计算接触区域中心
    contact_center = np.mean(np.asarray(pcd1.points)[contact_idx], axis=0)

    # 构建缩放变换
    T = np.eye(4)
    T[:3, 3] = -contact_center
    S = np.eye(4)
    S[:3, :3] *= scale
    T_inv = np.eye(4)
    T_inv[:3, 3] = contact_center

    # 应用约束缩放
    mesh2.transform(T_inv @ S @ T)

    return mesh1, mesh2

def visualize_results(mesh1, mesh2):
    """可视化结果"""
    mesh1.paint_uniform_color([0.9, 0.1, 0.1])  # 红色
    mesh2.paint_uniform_color([0.1, 0.1, 0.9])  # 蓝色
    o3d.visualization.draw_geometries([mesh1, mesh2])

def main():
    # 1. 加载并定位网格
    mesh1, mesh2 = load_and_position_meshes()

    # 2. 初始配准
    transform = feature_based_registration(mesh1, mesh2)

    # 3. 约束变形 (缩放系数0.7-1.3之间)
    mesh1, mesh2 = constrained_deformation(mesh1, mesh2, transform, scale=0.8)

    # 4. 精细ICP配准
    pcd1 = mesh1.sample_points_uniformly(1000)
    pcd2 = mesh2.sample_points_uniformly(1000)
    result = o3d.pipelines.registration.registration_icp(
        pcd1, pcd2, 0.1, np.eye(4),
        o3d.pipelines.registration.TransformationEstimationPointToPlane())
    mesh2.transform(result.transformation)

    # 5. 结果可视化与保存
    visualize_results(mesh1, mesh2)
    o3d.io.write_triangle_mesh("deformed_mesh1.obj", mesh1)
    o3d.io.write_triangle_mesh("deformed_mesh2.obj", mesh2)

if __name__ == "__main__":
    main()

[Open3D INFO] Skipping non-triangle primitive geometry of type: 2
[Open3D WARNING] Too few correspondences (4) after mutual filter, fall back to original correspondences.


In [ ]:
import open3d as o3d
import numpy as np
from pathlib import Path
from scipy.spatial import KDTree

# 定义平移和旋转
translation1 = [20, 0, 0]  # x, y, z 平移
rotation_matrix1 = o3d.geometry.get_rotation_matrix_from_axis_angle([0, np.radians(30), 0])  # 绕 y 轴旋转

translation2 = [-40, 0, 30]
rotation_matrix2 = o3d.geometry.get_rotation_matrix_from_axis_angle([0, np.radians(45), 0])  # 绕 y 轴旋转

# 加载网格文件
mesh1 = o3d.io.read_triangle_mesh(Path('../assets/basic_geo/3/3.obj'))
mesh2 = o3d.io.read_triangle_mesh(Path('../assets/basic_geo/2/2.obj'))

# 应用变换
mesh1.rotate(rotation_matrix1, center=(0, 0, 0))
mesh1.translate(translation1)

mesh2.rotate(rotation_matrix2, center=(0, 0, 0))
mesh2.translate(translation2)

# 从网格采样点云
pcd1 = mesh1.sample_points_uniformly(number_of_points=1000)
pcd2 = mesh2.sample_points_uniformly(number_of_points=1000)

# 计算法线
pcd1.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30))
pcd2.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30))

# 提取FPFH特征
fpfh1 = o3d.pipelines.registration.compute_fpfh_feature(
    pcd1, o3d.geometry.KDTreeSearchParamHybrid(radius=0.25, max_nn=100))
fpfh2 = o3d.pipelines.registration.compute_fpfh_feature(
    pcd2, o3d.geometry.KDTreeSearchParamHybrid(radius=0.25, max_nn=100))

# 使用RANSAC进行粗配准
distance_threshold = 0.5
result_ransac = o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
    pcd1, pcd2, fpfh1, fpfh2, mutual_filter=True,
    max_correspondence_distance=distance_threshold,
    estimation_method=o3d.pipelines.registration.TransformationEstimationPointToPoint(False),
    ransac_n=4,
    criteria=o3d.pipelines.registration.RANSACConvergenceCriteria(4000000, 500))

# 使用ICP进行精细配准
result_icp = o3d.pipelines.registration.registration_icp(
    pcd1, pcd2, max_correspondence_distance=0.2,
    init=result_ransac.transformation,
    estimation_method=o3d.pipelines.registration.TransformationEstimationPointToPlane())

# 应用ICP变换到点云
pcd2.transform(result_icp.transformation)

# 合并点云
merged_pcd = pcd1 + pcd2

# 可视化结果
o3d.visualization.draw_geometries([merged_pcd])

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
[Open3D INFO] Skipping non-triangle primitive geometry of type: 2
[Open3D WARNING] Too few correspondences (3) after mutual filter, fall back to original correspondences.


In [8]:
import open3d as o3d
import numpy as np
from pathlib import Path
from scipy.spatial import KDTree

# 定义平移和旋转
translation1 = [20, 0, 0]  # x, y, z 平移
rotation_matrix1 = o3d.geometry.get_rotation_matrix_from_axis_angle([0, np.radians(30), 0])  # 绕 y 轴旋转

translation2 = [-40, 0, 30]
rotation_matrix2 = o3d.geometry.get_rotation_matrix_from_axis_angle([0, np.radians(45), 0])  # 绕 y 轴旋转

# 加载网格文件
mesh1 = o3d.io.read_triangle_mesh(Path('../assets/basic_geo/3/3.obj'))
mesh2 = o3d.io.read_triangle_mesh(Path('../assets/basic_geo/2/2.obj'))

# 应用变换
mesh1.rotate(rotation_matrix1, center=(0, 0, 0))
mesh1.translate(translation1)

mesh2.rotate(rotation_matrix2, center=(0, 0, 0))
mesh2.translate(translation2)

# 从网格采样点云
pcd1 = mesh1.sample_points_uniformly(number_of_points=1000)
pcd2 = mesh2.sample_points_uniformly(number_of_points=1000)


# 检测相邻部分
kdtree = KDTree(np.asarray(pcd2.points))
distances, indices = kdtree.query(np.asarray(pcd1.points))
adjacent_indices = np.where(distances < 20.0)[0]  # 距离阈值为 5.0
# 输出结果
print(f"检测到的相邻点数量: {len(adjacent_indices)}")
print("最小距离:", np.min(distances))
print("最大距离:", np.max(distances))
print("平均距离:", np.mean(distances))
# 对相邻部分进行形变
if len(adjacent_indices) > 0:
    adjacent_points = np.asarray(pcd1.points)[adjacent_indices]
    center = np.mean(adjacent_points, axis=0)  # 相邻部分的中心
    print("相邻部分中心:", center)

    # 构建缩放矩阵
    scale_matrix = np.eye(4)
    scale_matrix[:3, :3] *= 1.2  # 缩放系数
    scale_matrix[:3, 3] = center * (1 - 1.2)  # 保持中心不变

    # 应用变换到 mesh2
    mesh2.transform(scale_matrix)
    print("形变已应用到 mesh2")
else:
    print("未检测到足够的相邻点，未进行形变")

# 可视化结果
o3d.visualization.draw_geometries([mesh1, mesh2])

[Open3D INFO] Skipping non-triangle primitive geometry of type: 2
检测到的相邻点数量: 176
最小距离: 12.536811249818072
最大距离: 77.70847190202605
平均距离: 45.384095684477664
相邻部分中心: [-4.49138278 20.93676128 16.4302877 ]
形变已应用到 mesh2


In [10]:
import open3d as o3d
import numpy as np
from pycpd import DeformableRegistration

# 加载网格并采样点云
def load_mesh_and_sample_points(file_path, num_points=1000):
    mesh = o3d.io.read_triangle_mesh(file_path)
    pcd = mesh.sample_points_uniformly(number_of_points=num_points)
    return mesh, pcd

# 定义目标路径点云
def create_target_path(num_points=1000):
    t = np.linspace(0, 2 * np.pi, num_points)
    x = 10 * np.sin(t)
    y = 10 * np.cos(t)
    z = np.linspace(0, 10, num_points)
    path_points = np.vstack((x, y, z)).T
    target_pcd = o3d.geometry.PointCloud()
    target_pcd.points = o3d.utility.Vector3dVector(path_points)
    return target_pcd

# 非刚性配准
def deform_mesh_to_path(mesh, source_pcd, target_pcd):
    source = np.asarray(source_pcd.points)
    target = np.asarray(target_pcd.points)

    def callback(iteration, error, X, Y):
        print(f"Iteration: {iteration}, Error: {error}")

    reg = DeformableRegistration(X=source, Y=target)
    transformed_source, _ = reg.register(callback)

    # 更新网格顶点
    mesh.vertices = o3d.utility.Vector3dVector(transformed_source)
    return mesh

# 主函数
def main():
    # 加载网格
    mesh1, pcd1 = load_mesh_and_sample_points('../assets/basic_geo/3/3.obj')
    mesh2, pcd2 = load_mesh_and_sample_points('../assets/basic_geo/2/2.obj')

    # 创建目标路径
    target_pcd = create_target_path()

    # 对网格1进行变形
    mesh1 = deform_mesh_to_path(mesh1, pcd1, target_pcd)

    # 对网格2进行变形
    mesh2 = deform_mesh_to_path(mesh2, pcd2, target_pcd)

    # 可视化结果
    mesh1.paint_uniform_color([1, 0, 0])  # 红色
    mesh2.paint_uniform_color([0, 0, 1])  # 蓝色
    o3d.visualization.draw_geometries([mesh1, mesh2, target_pcd])

if __name__ == "__main__":
    main()

[Open3D INFO] Skipping non-triangle primitive geometry of type: 2
Iteration: 1, Error: inf
Iteration: 2, Error: inf
Iteration: 3, Error: inf
Iteration: 4, Error: inf
Iteration: 5, Error: inf
Iteration: 6, Error: inf
Iteration: 7, Error: inf
Iteration: 8, Error: inf
Iteration: 9, Error: inf
Iteration: 10, Error: inf
Iteration: 1, Error: inf
Iteration: 2, Error: inf
Iteration: 3, Error: inf
Iteration: 4, Error: inf
Iteration: 5, Error: inf
Iteration: 6, Error: inf
Iteration: 7, Error: inf
Iteration: 8, Error: inf
Iteration: 9, Error: inf
Iteration: 10, Error: inf
Iteration: 11, Error: inf
Iteration: 12, Error: inf


In [16]:
import open3d as o3d
import numpy as np
from pycpd import DeformableRegistration

# 加载网格并采样点云
def load_mesh_and_sample_points(file_path, num_points=1000):
    mesh = o3d.io.read_triangle_mesh(file_path)
    pcd = mesh.sample_points_uniformly(number_of_points=num_points)
    return mesh, pcd

# 定义目标折线路径点云
def create_target_path():
    points = np.array([
        [0, 0, 0],
        [20, 0, 0],
        [20, 0, 30],
        [40, 0, 50]
    ])
    target_pcd = o3d.geometry.PointCloud()
    target_pcd.points = o3d.utility.Vector3dVector(points)
    return target_pcd

# 非刚性配准
def deform_mesh_to_path(mesh, source_pcd, target_pcd):
    source = np.asarray(source_pcd.points)
    target = np.asarray(target_pcd.points)

    def callback(iteration, error, X, Y):
        print(f"Iteration: {iteration}, Error: {error}")

    reg = DeformableRegistration(X=source, Y=target)
    transformed_source, _ = reg.register(callback)

    # 更新网格顶点
    mesh.vertices = o3d.utility.Vector3dVector(transformed_source)
    return mesh

# 主函数
def main():
    # 加载网格
    mesh1, pcd1 = load_mesh_and_sample_points('../assets/basic_geo/3/3.obj')
    mesh2, pcd2 = load_mesh_and_sample_points('../assets/basic_geo/2/2.obj')

    # 创建目标路径
    target_pcd = create_target_path()

    # 对网格1进行变形
    mesh1 = deform_mesh_to_path(mesh1, pcd1, target_pcd)

    # 对网格2进行变形
    mesh2 = deform_mesh_to_path(mesh2, pcd2, target_pcd)

    # 可视化结果
    mesh1.paint_uniform_color([1, 0, 0])  # 红色
    mesh2.paint_uniform_color([0, 1, 0])  # 绿色
    o3d.visualization.draw_geometries([mesh1, mesh2, target_pcd])

if __name__ == "__main__":
    main()

[Open3D INFO] Skipping non-triangle primitive geometry of type: 2
Iteration: 1, Error: inf
Iteration: 2, Error: inf
Iteration: 3, Error: inf
Iteration: 4, Error: inf
Iteration: 5, Error: inf
Iteration: 6, Error: inf
Iteration: 7, Error: inf
Iteration: 8, Error: inf
Iteration: 9, Error: inf
Iteration: 10, Error: inf
Iteration: 11, Error: inf
Iteration: 12, Error: inf
Iteration: 13, Error: inf
Iteration: 14, Error: inf
Iteration: 15, Error: inf
Iteration: 16, Error: inf
Iteration: 17, Error: inf
Iteration: 18, Error: inf
Iteration: 19, Error: inf
Iteration: 20, Error: inf
Iteration: 21, Error: inf
Iteration: 22, Error: inf
Iteration: 23, Error: inf
Iteration: 24, Error: inf
Iteration: 25, Error: inf
Iteration: 26, Error: inf
Iteration: 27, Error: inf
Iteration: 1, Error: inf
Iteration: 2, Error: inf
Iteration: 3, Error: inf
Iteration: 4, Error: inf
Iteration: 5, Error: inf
Iteration: 6, Error: inf
Iteration: 7, Error: inf
Iteration: 8, Error: inf
Iteration: 9, Error: inf
Iteration: 10, E